# 02.5 — Autograd basics

Autograd is the reason PyTorch exists: you write the *forward* computation, and PyTorch derives the gradients automatically. This is what MATLAB's `dlfeval` + `dlgradient` do — but PyTorch's version is always-on, works on any Python code, and is worth understanding mechanically because **every training bug eventually touches it**.

This notebook covers:

* `requires_grad`, the computational graph, and `.backward()`.
* Verifying autograd against finite differences (trust, but check).
* Why gradients **accumulate** — the bug and the feature.
* `no_grad`, `detach`, `.item()` — leaving the graph on purpose.
* Freezing parameters with `requires_grad_(False)`.

**Prerequisite:** [02.4 PyTorch tensors](02.4_pytorch_tensors_intro.ipynb).

## Section 1 — What MATLAB does

In MATLAB's Deep Learning Toolbox, gradients only exist inside a function traced by `dlfeval`:

```matlab
function [loss, gradients] = modelGradients(net, dlX, T)
    dlY = forward(net, dlX);
    loss = crossentropy(dlY, T);
    gradients = dlgradient(loss, net.Learnables);   % ← derivatives requested here
end

% ...in the training loop:
[loss, gradients] = dlfeval(@modelGradients, net, dlX, T);
net = adamupdate(net, gradients, ...);
```

Three MATLAB facts to carry over: the data must be a `dlarray`, the tracing happens only inside `dlfeval`, and `dlgradient` names *which* derivative you want (loss w.r.t. learnables).

PyTorch's answer to all three: tensors carry a `requires_grad` flag, **every** operation on them is traced (no special context needed), and `loss.backward()` computes d(loss)/d(everything-that-requires-grad) in one call.

## Section 2 — The Python concepts you need

### 2.1 — `requires_grad` and the graph

Setting `requires_grad=True` on a tensor makes PyTorch record every operation that touches it, building a **computational graph** as your code runs:

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x        # graph: x → square → (+) ← times-3 ← x
print(y)                 # note grad_fn=<AddBackward0> — y knows how it was made

In [ ]:
y.backward()             # walk the graph backwards, filling in .grad
print(x.grad)            # dy/dx = 2x + 3 = 7 at x=2  ✓

The graph is **dynamic** — rebuilt on every forward pass, from whatever Python actually executed (including `if`s and loops). That's the deep difference from MATLAB's `layerGraph`, which is a static structure you assemble up front.

A picture of what just happened:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 4))

def node(x, y, label, sub, color):
    rect = mpatches.FancyBboxPatch((x - 0.75, y - 0.42), 1.5, 0.84,
        boxstyle="round,pad=0.06", facecolor=color, edgecolor="black", linewidth=1.3)
    ax.add_patch(rect)
    ax.text(x, y + 0.14, label, ha="center", fontsize=12, family="monospace", fontweight="bold")
    ax.text(x, y - 0.18, sub, ha="center", fontsize=9, color="#444")

node(1.0, 2.0, "x = 2.0", "leaf\nrequires_grad", "#ffe4cc")
node(4.0, 3.1, "x**2", "PowBackward", "#cce4ff")
node(4.0, 0.9, "3 * x", "MulBackward", "#cce4ff")
node(7.0, 2.0, "y = 10.0", "AddBackward\n(output)", "#e6f0d0")

for x1, y1, x2, y2 in [(1.75, 2.25, 3.25, 3.0), (1.75, 1.75, 3.25, 1.0),
                        (4.75, 3.0, 6.25, 2.25), (4.75, 1.0, 6.25, 1.75)]:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color="black", lw=1.4))

# backward flow
ax.annotate("", xy=(1.45, 1.35), xytext=(6.55, 1.35),
    arrowprops=dict(arrowstyle="->", color="#aa0000", lw=2.2,
                    connectionstyle="arc3,rad=0.35"))
ax.text(4.0, -0.15, "y.backward()  —  chain rule flows right-to-left, deposits dy/dx into x.grad",
    ha="center", fontsize=11, color="#aa0000")

ax.set_xlim(-0.1, 8.2); ax.set_ylim(-0.6, 4.0)
ax.axis("off")
ax.set_title("The dynamic computational graph for  y = x² + 3x", fontsize=12)
plt.tight_layout(); plt.show()

### 2.2 — Trust, but verify: finite differences

Autograd is exact (to floating point), but *checking* a gradient numerically is a habit worth having — it's how you debug custom loss functions, and this project's parity tests use the same verify-empirically philosophy:

In [ ]:
def f(x):
    return x**2 + 3 * x

x0 = torch.tensor(2.0, requires_grad=True)
f(x0).backward()

# Finite differences in float64 — in float32, subtracting two nearly-equal
# values loses half the significant digits ("catastrophic cancellation"),
# and the check would read 6.995 instead of 7. gradcheck uses double for
# exactly this reason.
eps = 1e-4
xd = torch.tensor(2.0, dtype=torch.float64)
numeric = (f(xd + eps) - f(xd - eps)) / (2 * eps)
print(f"autograd:            {x0.grad.item():.6f}")
print(f"finite differences:  {numeric.item():.6f}")

(For real models, `torch.autograd.gradcheck` automates this comparison over every input, in float64.)

### 2.3 — Gradients ACCUMULATE — the bug and the feature

`backward()` doesn't *assign* to `.grad`; it **adds** to it. Two backward calls without clearing = doubled gradients:

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

(w * 3).backward()
print("after 1st backward:", w.grad)     # 3

(w * 3).backward()
print("after 2nd backward:", w.grad)     # 6 — accumulated, NOT replaced!

w.grad = None                             # what optimizer.zero_grad() does since PyTorch 2.0
(w * 3).backward()
print("after clearing:    ", w.grad)     # 3 again

**The bug**: forget to clear between training steps and every step applies the sum of all past gradients — the model "trains" but behaves bizarrely. Hence the sacred order in every PyTorch loop:

```python
optimizer.zero_grad()     # 1. clear
loss.backward()           # 2. compute
optimizer.step()          # 3. apply
```

**The feature**: this codebase *exploits* accumulation deliberately. Hardware-aware gradient accumulation (`training/accumulation.py`, Critical Note #18) splits a too-big-for-GPU batch into micro-batches, calls `backward()` on each **without clearing**, and steps once at the end — the accumulated sum is mathematically the full-batch gradient. Same mechanism; intent is the only difference.

### 2.4 — Leaving the graph on purpose

Three tools for "I want the value, not the history":

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2

# .item() — extract a Python scalar (for printing/logging); no graph attached
print("loss value for logging:", y.item())

# .detach() — a tensor view sharing data but cut out of the graph
y_free = y.detach()
print("detached:", y_free.requires_grad)

# torch.no_grad() — a context where NOTHING is traced (evaluation loops)
with torch.no_grad():
    z = x * 10
print("built under no_grad:", z.requires_grad)

When each appears in a real training loop:

| Tool | Where you'll see it |
|---|---|
| `.item()` | logging: `history.append(loss.item())` — keeping a graph-attached tensor in a list leaks the entire graph's memory every iteration |
| `.detach()` | feeding a model output into something that must not backprop (e.g. metrics, plots: `pred.detach().cpu().numpy()`) |
| `torch.no_grad()` | validation/test loops — no gradients needed, ~half the memory, faster |

### 2.5 — Freezing: `requires_grad_(False)`

Every `nn.Module` parameter has `requires_grad=True` by default. Flipping it off **freezes** the parameter — no gradient computed, optimizer can't move it:

In [ ]:
import torch.nn as nn

layer = nn.Linear(4, 2)
layer.weight.requires_grad_(False)         # freeze the weights, leave bias trainable

out = layer(torch.randn(3, 4)).sum()
out.backward()
print("weight.grad:", layer.weight.grad)   # None — frozen
print("bias.grad:  ", layer.bias.grad)     # populated

This one flag is the foundation of the project's **freeze/unfreeze curriculum** (`training/freezing.py`): the Soft Three-Stage Curriculum thaws the encoder, decoder, and classifier on different epoch schedules — implemented as exactly this toggle, driven per-module per-epoch. The deep dive is planned as notebook 07.5.

## Section 3 — The `neural_data_decoding` implementation

The production training loop's backward pass — one `backward()` on ONE total loss, even though that loss sums reconstruction + KL + classification + confidence (Critical Note #28: a single gradient root distributes to all three subnetworks through the graph, exactly like the `+` node in our figure):

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/loop.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if ".backward()" in line)
for k in range(max(0, i - 6), min(i + 10, len(src))):
    marker = " ►" if ".backward()" in src[k] else "  "
    print(f"{k + 1:4}{marker}| {src[k]}")

Things to spot:

* One `total_loss.backward()` — NOT separate backwards per loss component. The graph routes each component's gradient to the parameters it touched; summing first is both simpler and correct.
* Gradient clipping right after `backward()` and before `step()` — the only window where `.grad` is populated and not yet applied.
* No manual derivative code anywhere in the file. The MATLAB port replaced `cgg_procGradientAggregation` + `dlgradient` plumbing with the graph doing the bookkeeping.

## Section 4 — Hands-on exercises

### Exercise 4.1 — predict, then verify

For `y = x³` at `x = 2`, what will `x.grad` be after TWO backward calls without clearing? Predict, then run.

In [ ]:
# Your prediction: ___
x = torch.tensor(2.0, requires_grad=True)
(x**3).backward()
(x**3).backward()
print(x.grad)     # dy/dx = 3x² = 12, twice → 24

### Exercise 4.2 — a two-parameter gradient

Compute d(loss)/da and d(loss)/db for `loss = (a*b + a)²` at `a=1, b=2`, and verify one of them by hand. (By the chain rule: `loss = (a(b+1))²`, so `d/da = 2a(b+1)² = 18`, `d/db = 2a²(b+1) = 6`.)

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
a = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
loss = (a * b + a)**2
loss.backward()
print(f"d/da = {a.grad.item()}, d/db = {b.grad.item()}")

## Section 5 — Common errors

### `RuntimeError: grad can be implicitly created only for scalar outputs`

You called `.backward()` on a non-scalar (e.g. a whole batch of losses). Reduce first — `loss.mean().backward()` — which is almost always what you meant.

### `RuntimeError: Trying to backward through the graph a second time`

The graph is freed after `backward()` by default. Calling backward twice on the *same* forward pass needs `retain_graph=True` — but wanting that is usually a sign the loop structure is wrong (recompute the forward instead).

### `.grad` is `None` after backward

Three causes in order: the tensor never had `requires_grad=True`; it's not a **leaf** (gradients only accumulate on leaves — intermediates need `retain_grad()`); or the op ran under `no_grad()`.

### `RuntimeError: Can't call numpy() on Tensor that requires grad`

Exactly what it says — detach first: `t.detach().cpu().numpy()`.

### Loss goes weird after a few steps (no exception)

Check for a missing `zero_grad()` — accumulated gradients from prior steps are being applied. The silent version of §2.3.

### `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`

The in-place `_` methods from [02.4 §2.5](02.4_pytorch_tensors_intro.ipynb) colliding with the graph. Use the out-of-place form, or wrap intentional parameter surgery in `with torch.no_grad():`.

## Section 6 — Further reading

- [Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html) — the official deep dive.
- [A gentle introduction to autograd](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html) — the tutorial with pictures.
- [`torch.autograd.gradcheck`](https://pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html) — automated finite-difference verification.

Next up: **[02.6 nn.Module vs layerGraph](02.6_nn_module_vs_layergraph.ipynb)**.